In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('Thursday.csv')
df = data.copy()

In [ ]:
df['Label'] = df['Label'].map(lambda x: 0 if x == "BENIGN" else 1)


print("Unikalne wartości w kolumnie 'Label':")
print(df['Label'].unique())
print(df['Label'].value_counts())

df.columns = df.columns.str.replace(' ', '_')
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

In [ ]:
X = df.drop('Label', axis=1).values
y = df['Label'].values

In [ ]:
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42, stratify=y)

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'Precision', 'Recall'])

model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

y_pred_probs = model.predict(X_test)
y_pred_classes = (y_pred_probs >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred_classes)
print(cm)
print(classification_report(y_test, y_pred_classes, target_names=['BENIGN', 'ATTACK']))

Confusion Matrix
True Negatives (TN): 90,979 — poprawnie sklasyfikowane BENIGN.
False Positives (FP): 304 — BENIGN błędnie sklasyfikowane jako ATTACK.
False Negatives (FN): 19 — ATTACK błędnie sklasyfikowane jako BENIGN.
True Positives (TP): 424 — poprawnie wykryte ATTACK

Metryki dla klasy ATTACK
Precision: 0.58

Tylko 58% próbek przewidzianych jako ATTACK jest rzeczywiście atakami.
Model generuje stosunkowo dużo fałszywych alarmów (FP = 304).
Recall: 0.96

Model wykrywa 96% wszystkich ataków (ATTACK), co oznacza, że bardzo rzadko pomija rzeczywiste ataki (FN = 19).
F1-score: 0.72

Jest to harmoniczna średnia precision i recall. Wynik 0.72 wskazuje, że model ma problemy z precyzją, ale dobrze radzi sobie z wykrywaniem ataków.
Ogólne metryki
Accuracy: 1.00

Wysoka dokładność wynika z dominacji klasy BENIGN w zbiorze danych. Model dobrze radzi sobie z klasyfikacją tej klasy, co zawyża accuracy.
Macro avg:

Precision: 0.79
Średnia precyzja dla obu klas.
Recall: 0.98
Średnia recall dla obu klas.
F1-score: 0.86
Średnia F1-score dla obu klas.
Weighted avg:

Wartości ważone proporcjonalnie do liczby próbek w każdej klasie. Wysokie wyniki wynikają z dominacji klasy BENIGN.
Interpretacja wyników
Klasa BENIGN:

Model działa niemal idealnie dla klasy BENIGN (precision, recall i F1-score wynoszą 1.00).
Klasa ATTACK:

Recall (0.96): Model bardzo dobrze wykrywa ataki, co oznacza, że rzadko pomija rzeczywiste ataki (FN = 19).
Precision (0.58): Model ma problemy z precyzją, co oznacza, że generuje stosunkowo dużo fałszywych alarmów (FP = 304).
F1-score (0.72): Wynik wskazuje na nierównowagę między precision i recall.
Ogólna dokładność (1.00):

Wysoka dokładność wynika z dominacji klasy BENIGN. Nie jest to najlepsza metryka w przypadku niezbalansowanych danych.
Co oznaczają te wyniki w praktyce?
Zalety:

Model bardzo dobrze wykrywa ataki (Recall = 0.96), co jest kluczowe w systemach wykrywania intruzów.
Model rzadko pomija rzeczywiste ataki (FN = 19), co oznacza, że jest skuteczny w ochronie systemu.
Wady:

Model generuje stosunkowo dużo fałszywych alarmów (FP = 304), co może być problematyczne w praktyce, ponieważ fałszywe alarmy mogą obciążać systemy monitorujące.

[[90979   304]
 [   19   424]]
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     91283
      ATTACK       0.58      0.96      0.72       443

    accuracy                           1.00     91726
   macro avg       0.79      0.98      0.86     91726
weighted avg       1.00      1.00      1.00     91726
